In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3433.53it/s]


In [20]:
import os
files = [
    "data/raw/egg.txt",
    "data/raw/milk.txt",
    "data/raw/storage.txt",
    "data/raw/fruits_and_vegetable.txt"
]

paragraphs = []
documents = []
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    paras = [
        p.strip()
        for p in text.split("\n\n")
        if p.strip()
    ]
    title = os.path.splitext(os.path.basename(file))[0]
    
    for para in paras:
        documents.append({
            "title": title,
            "text": para,
            "source": file
        })
        
    paragraphs.extend(paras)
print(len(paragraphs))
print(paragraphs[0])
print(paragraphs[-1])

print(documents[0])

133
Egg
Keep fruit easy to reach;
Take fruits and vegetables with you to have as snacks;
(re)discover new or forgotten vegetables;
Check what is in season where you are (& try new recipes);
Swap your old favourites to increase variety.
{'title': 'egg', 'text': 'Egg', 'source': 'data/raw/egg.txt'}


In [21]:
texts = [doc["text"] for doc in documents]

embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:03<00:00,  1.52it/s]


In [22]:
embeddings.shape

(133, 384)

In [23]:
import numpy as np
threshold = 0.70
max_words = 350
min_chunk_size = 200

chunks = []

current_chunk = {
    "id": 0,
    "title": documents[0]["title"],
    "text": documents[0]["text"],
    "source": documents[0]["source"]
}

for i in range(len(documents) - 1):

    e_i = embeddings[i]
    e_j = embeddings[i + 1]

    similarity = e_i.dot(e_j) / (
        np.linalg.norm(e_i) * np.linalg.norm(e_j)
    )

    current_word_count = len(current_chunk["text"].split())
    next_word_count = len(documents[i + 1]["text"].split())

    same_source = (
        current_chunk["source"] == documents[i + 1]["source"]
    )

    if (
        similarity >= threshold
        and same_source
        and current_word_count + next_word_count <= max_words
    ):

        current_chunk["text"] += "\n\n" + documents[i + 1]["text"]

    else:

        chunks.append(current_chunk)

        current_chunk = {
            "id": len(chunks),
            "title": documents[i + 1]["title"],
            "text": documents[i + 1]["text"],
            "source": documents[i + 1]["source"]
        }

# Save final chunk
chunks.append(current_chunk)


# -------------------------
# Second pass: merge small chunks
# -------------------------
merged_chunks = []

for chunk in chunks:

    word_count = len(chunk["text"].split())

    if merged_chunks:

        previous_word_count = len(merged_chunks[-1]["text"].split())

        if (
            word_count < min_chunk_size
            and merged_chunks[-1]["source"] == chunk["source"]
            and previous_word_count + word_count <= max_words
        ):

            merged_chunks[-1]["text"] += "\n\n" + chunk["text"]

        else:
            merged_chunks.append(chunk)

    else:
        merged_chunks.append(chunk)

chunks = merged_chunks


# -------------------------
# Reassign IDs
# -------------------------

for i, chunk in enumerate(chunks):
    chunk["id"] = i

In [24]:
len(chunks)

39

In [25]:
for i, chunk in enumerate(chunks):
    words = len(chunk["text"].split())
    print(f"Chunk {i}: {words} words")

Chunk 0: 245 words
Chunk 1: 139 words
Chunk 2: 218 words
Chunk 3: 283 words
Chunk 4: 347 words
Chunk 5: 237 words
Chunk 6: 331 words
Chunk 7: 325 words
Chunk 8: 294 words
Chunk 9: 227 words
Chunk 10: 347 words
Chunk 11: 197 words
Chunk 12: 212 words
Chunk 13: 305 words
Chunk 14: 173 words
Chunk 15: 296 words
Chunk 16: 336 words
Chunk 17: 258 words
Chunk 18: 278 words
Chunk 19: 345 words
Chunk 20: 336 words
Chunk 21: 320 words
Chunk 22: 67 words
Chunk 23: 271 words
Chunk 24: 166 words
Chunk 25: 204 words
Chunk 26: 295 words
Chunk 27: 317 words
Chunk 28: 343 words
Chunk 29: 328 words
Chunk 30: 71 words
Chunk 31: 184 words
Chunk 32: 252 words
Chunk 33: 310 words
Chunk 34: 298 words
Chunk 35: 230 words
Chunk 36: 249 words
Chunk 37: 327 words
Chunk 38: 189 words


In [27]:
for chunk in chunks:

    text = chunk["text"]

    embedding = model.encode(
        text,
        convert_to_numpy=True
    )

    chunk["embedding"] = embedding.tolist()

In [29]:
import json

with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(
        chunks,
        f,
        indent=4,
        ensure_ascii=False
    )

In [28]:
print(type(chunks))
print(type(chunks[0]))
print(chunks[0].keys())

<class 'list'>
<class 'dict'>
dict_keys(['id', 'title', 'text', 'source', 'embedding'])
